In [1]:
from pathlib import Path
import json, re

folder = Path("/path/1_generate_qa/1_qa_file_output_folder")
files = sorted(
    folder.glob("part_*.jsonl"),
    key=lambda f: int(re.search(r"part_(\d+)\.jsonl", f.name).group(1))
)

all_records = []
for fp in files:
    with fp.open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            all_records.append(json.loads(line))



In [2]:
all_records[:2]

[{'caption': '#### 1. **Which vehicle in the image has a blue body with red wheels?**\n   - A) The green pickup truck\n   - B) The yellow car\n   - C) The blue car\n   - D) The gray car\n\n**Answer:** C) The blue car  \n------\n#### 2. **How many bicycles are present in the image?**\n   - A) One\n   - B) Two\n   - C) Three\n   - D) Four\n\n**Answer:** B) Two  \n------\n#### 3. **What type of aircraft is depicted in the image?**\n   - A) Helicopter\n   - B) Jet plane\n   - C) Propeller plane\n   - D) Fighter jet\n\n**Answer:** D) Fighter jet  \n------\n#### 4. **Which vehicle is positioned closest to the fighter jet?**\n   - A) The yellow car\n   - B) The green pickup truck\n   - C) The blue car\n   - D) The gray car\n\n**Answer:** A) The yellow car  \n------\n#### 5. **What is the color of the motorcycle in the image?**\n   - A) Green\n   - B) Red\n   - C) Blue\n   - D) Yellow\n\n**Answer:** A) Green  \n------',
  'image_path': 'h_cluster:s3://mllm/internvl_data/internvl_data_image_vid

In [5]:
def extract_questions_and_answers(text):
    # split text into blocks using '------' as the delimiter
    blocks = text.split('------')
    result = []
    
    for block in blocks:
        # skip empty blocks
        if not block.strip():
            continue
            
        lines = block.strip().splitlines()
        # ensure there are at least two lines (question and answer)
        if len(lines) < 2:
            continue
            
        # merge all lines except the last one as the question
        question_block = '\n'.join(lines[:-1]).strip()
        # extract the answer from the last line
        answer_line = lines[-1].strip()
        answer = answer_line.replace("**Answer:** ", "").strip()
        question_block = question_block[10:].replace('**', '')
        result.append({
            "question": question_block,
            "answer": answer[0]
        })
    
    return result

extract_questions_and_answers(all_records[0]['caption'])

[{'question': 'Which vehicle in the image has a blue body with red wheels?\n   - A) The green pickup truck\n   - B) The yellow car\n   - C) The blue car\n   - D) The gray car',
  'answer': 'C'},
 {'question': 'How many bicycles are present in the image?\n   - A) One\n   - B) Two\n   - C) Three\n   - D) Four',
  'answer': 'B'},
 {'question': 'What type of aircraft is depicted in the image?\n   - A) Helicopter\n   - B) Jet plane\n   - C) Propeller plane\n   - D) Fighter jet',
  'answer': 'D'},
 {'question': 'Which vehicle is positioned closest to the fighter jet?\n   - A) The yellow car\n   - B) The green pickup truck\n   - C) The blue car\n   - D) The gray car',
  'answer': 'A'},
 {'question': 'What is the color of the motorcycle in the image?\n   - A) Green\n   - B) Red\n   - C) Blue\n   - D) Yellow',
  'answer': 'A'}]

In [6]:
extract_qa=[]
for sample in all_records:
    extract_qa.append({
        "image_path":sample['image_path'],
        "qa_list":extract_questions_and_answers(sample["caption"])
    })


In [8]:
len(extract_qa),extract_qa[0]

(300000,
 {'image_path': 'h_cluster:s3://mllm/internvl_data/internvl_data_image_video/langchao2other/multi-modal/Super-CLEVR/images/superCLEVR_new_004806.png',
  'qa_list': [{'question': 'Which vehicle in the image has a blue body with red wheels?\n   - A) The green pickup truck\n   - B) The yellow car\n   - C) The blue car\n   - D) The gray car',
    'answer': 'C'},
   {'question': 'How many bicycles are present in the image?\n   - A) One\n   - B) Two\n   - C) Three\n   - D) Four',
    'answer': 'B'},
   {'question': 'What type of aircraft is depicted in the image?\n   - A) Helicopter\n   - B) Jet plane\n   - C) Propeller plane\n   - D) Fighter jet',
    'answer': 'D'},
   {'question': 'Which vehicle is positioned closest to the fighter jet?\n   - A) The yellow car\n   - B) The green pickup truck\n   - C) The blue car\n   - D) The gray car',
    'answer': 'A'},
   {'question': 'What is the color of the motorcycle in the image?\n   - A) Green\n   - B) Red\n   - C) Blue\n   - D) Yellow'

In [9]:
import json
json.dump(extract_qa, open('/path/2_extract_qa/2_output_sample.json', 'w'))